0.ライブラリなどの環境構築

In [ ]:
%pip install pandas numpy datasets matplotlib scikit-learn lightgbm catboost

In [ ]:
import pandas as pd
import numpy as np
import catboost as cb
import lightgbm as lgb

from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


1.データの読み込み

In [ ]:
raw_dataset = load_dataset(
    "parquet", 
    data_files="hf://datasets/mstz/mushroom@refs/convert/parquet/mushroom/train/*.parquet"
)

In [ ]:
# データセットをdataframeに変換
df_raw = raw_dataset["train"].to_pandas()

2.前処理の定義

2.1onehotencodingでの前処理

In [ ]:
# カラムのほとんどがカテゴリ変数で、objectなので、object以外を明示する
str_bool_col_name = "has_bruises"
y_col = "is_poisonous"

# has_bruises bool
# is_poisonous int64

In [ ]:
def preprocessing_svm(df):
    
    # onehotencoding
    # カテゴリー型に変更→onehotencdingの順番で実施
    list_category_cols = []
    
    # for col in df.columns:
    #     if col not in (str_bool_col_name, y):
    #         df[col] = df[col].astype("category")
    #         list_category_cols.append(col)

    # ほぼ正解データに近いodorを除外してみる
    for col in df.columns:
        if col not in (str_bool_col_name, y, "odor"):
            df[col] = df[col].astype("category")
            list_category_cols.append(col)

    df = pd.get_dummies(df, columns=list_category_cols)

    # binary(yes or no)のカラムを0,1フラグに変換
    # df[str_bool_col_name] = df[str_bool_col_name].str.lower().map({"True": 1, "False": 0})

    # return df
    return df.drop("odor", axis=1)
    

In [ ]:
# 前処理関数の動作確認
df_tmp = df_raw.copy()

df_tmp = preprocessing_svm(df_tmp)

In [ ]:
df_tmp.dtypes

3.モデル学習

3.1モデル学習

In [ ]:
def onehot_train_and_evaluation(df_train, df_test, y_col, seed):

    # データセットの作成
    X = df_train.drop(y_col, axis=1)
    x_cols = X.columns
    y = df_train[y_col]

    X_train, X_val, y_train, y_val = train_test_split(X, 
                                                      y, 
                                                      test_size=0.2, 
                                                      random_state=seed
                                                     )

    X_test = df_test.drop(y_col, axis=1)
    y_test = df_test[y_col]

    # boosting系にevalsetを作成
    eval_set = [(X_val.reset_index(drop=True), y_val.reset_index(drop=True))]

    # モデルのインスタンス作成
    model_lgb = lgb.LGBMClassifier(objective="binary", random_state=seed, verbose=-1)
    model_cb = cb.CatBoostClassifier(loss_function="Logloss", random_seed=seed)
    model_svm = SVC(kernel="rbf", random_state=seed, verbose=False)  #確率出さない

    print("model_lgb train starts")
    model_lgb.fit(
        X_train, 
        y_train,
        eval_set = eval_set,
        callbacks=[
        lgb.early_stopping(stopping_rounds=50)
        ]
    )

    print("model_cb train starts")
    model_cb.fit(
        X_train, 
        y_train,
        eval_set = eval_set,
        early_stopping_rounds=50,
        verbose = False
    )

    print("model_svm train starts")
    model_svm.fit(
        X_train, 
        y_train,
    )

    # 予測
    y_pred_lgb = model_lgb.predict(X_test)

    y_pred_cb = model_cb.predict(X_test)

    y_pred_svm = model_svm.predict(X_test)

    # 結果出力用のDataFrame
    df = pd.DataFrame()

    df["model_lgb"] = [
        accuracy_score(y_test, y_pred_lgb),
        recall_score(y_test, y_pred_lgb, pos_label=True),
        precision_score(y_test, y_pred_lgb, pos_label=True),
        f1_score(y_test, y_pred_lgb, pos_label=True)
    ]

    df["model_cb"] = [
        accuracy_score(y_test, y_pred_cb),
        recall_score(y_test, y_pred_cb, pos_label=True),
        precision_score(y_test, y_pred_cb, pos_label=True),
        f1_score(y_test, y_pred_cb, pos_label=True)
    ]

    df["model_svm"] = [
        accuracy_score(y_test, y_pred_svm),
        recall_score(y_test, y_pred_svm, pos_label=True),
        precision_score(y_test, y_pred_svm, pos_label=True),
        f1_score(y_test, y_pred_svm, pos_label=True)
    ]

    df.index = ["acc", "recall", "precision", "f1"]


    # return df

    # すべての評価指標が1になったのでリークの確認
    # return X.columns.to_list()

    return pd.Series(
        model_lgb.feature_importances_,
        index=X_train.columns
    ).sort_values(ascending=False).head(20)
    

In [ ]:
seed = 42
df_data = df_raw.copy()

df_data = preprocessing_svm(df_data)
df_train, df_test = train_test_split(df_data, test_size=0.2, random_state=seed)

In [ ]:
df_scores_onehot = onehot_train_and_evaluation(
                            df_train=df_train, 
                            df_test=df_test, 
                            y_col=y_col,
                            seed=seed
                           )

In [ ]:
df_scores_onehot

# 全ての指標が１になった

3.2全ての評価指標が１になった原因分析

In [ ]:
onehot_train_and_evaluation(
                            df_train=df_train, 
                            df_test=df_test, 
                            y_col=y_col,
                            seed=seed
                           )

In [ ]:
# クロス集計
pd.crosstab(df_raw["odor"], df_raw[y_col])

# ほぼodorだけで、分類できてしまう。